# Fetch candidate models + MARS-LVIG and re-host them on your Kaggle account

Downloads each item from its original host (Hugging Face / Google Drive), checks it, uploads it to **your** Kaggle account, then deletes the local copy before the next item - so the notebook never needs more disk than the single largest item (~9.1 GB).

- Models go up as **Kaggle Models** (`<you>/<model>/pyTorch/<variation>`), MARS-LVIG as a **Kaggle Dataset**. `kagglehub` creates both **private** - keep them that way: VGGT-1B, VGGT-Omega and MARS-LVIG are non-commercial licences.
- Every `.pt` checkpoint is scanned (without loading it) for what its pickle would import; anything beyond `torch.*` / `collections.*` aborts that upload.

**Before running**
1. Settings -> **Internet: On**. No GPU needed (use the CPU accelerator to save GPU quota).
2. **Add-ons -> Secrets**, add and attach to this notebook:
   - `KAGGLE_USERNAME`, `KAGGLE_KEY` - from kaggle.com -> Settings -> API -> *Create New Token* (the downloaded `kaggle.json` holds both). Required: uploads need your own credentials.
   - `HF_TOKEN` - only for **official** VGGT-Omega, which is gated: request access at https://huggingface.co/facebook/VGGT-Omega first, then use a token from an account that was approved.

**About the `wincentIsMe/VGGT-Omega` mirror** (checked 2026-09-30): its `vggt_omega_1b_512.pt` is *not* byte-identical to Meta's despite its README saying "unmodified" - 4,576,706,322 vs 4,576,706,117 bytes, SHA-256 `6742f744...` vs `c02da418...`. Its pickle only imports standard `torch`/`collections` types (no code-execution payload), and the size difference is consistent with a load-and-resave, but the tensor values are unverified. It is therefore opt-in below, labelled in the upload's version notes, and section 5 can compare it tensor-by-tensor against the official file once you have access.

In [1]:
!pip install -q -U kagglehub huggingface_hub gdown

import os
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()

def secret(name):
    try:
        return _secrets.get_secret(name)
    except Exception:
        return None

for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    v = secret(k)
    assert v, "Add-ons -> Secrets: add %s (kaggle.com -> Settings -> API -> Create New Token) and attach it" % k
    os.environ[k] = v
HF_TOKEN = secret("HF_TOKEN")
USER = os.environ["KAGGLE_USERNAME"]

import kagglehub
kagglehub.whoami()
print("HF_TOKEN:", "set" if HF_TOKEN else "not set (fine unless you want official VGGT-Omega)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 817.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cl

## 1. Choose what to fetch

Sizes are the weight files on Hugging Face / Google Drive (2026-09-30). Remove keys from `RUN_MODELS` / `RUN_DATASETS` to skip them. Re-running an item uploads a **new version** of the same Kaggle model/dataset.

| key | source | size | licence |
|---|---|---|---|
| `vggt-1b` | facebook/VGGT-1B `model.pt` | 5.03 GB | CC BY-NC 4.0 |
| `vggt-omega-512` | facebook/VGGT-Omega (gated) or mirror | 4.58 GB | FAIR Noncommercial Research |
| `da3-small` / `da3-base` | depth-anything/DA3-SMALL / DA3-BASE | 0.14 / 0.54 GB | Apache 2.0 |
| `da3-large-1.1` / `da3-metric-large` | depth-anything/DA3-LARGE-1.1 / DA3METRIC-LARGE | 1.64 / 1.34 GB | Apache 2.0 |
| `mapanything-apache` | facebook/map-anything-apache | 4.91 GB | Apache 2.0 |
| `mars-lvig-hkairport-gnss03` | MARS-LVIG `HKairport_GNSS03.bag` (Google Drive) | 9.09 GB | CC BY-NC-SA 4.0 |

`vggt-1b` is uploaded as `vggt1b_model.pt` under `<you>/vggt1b-rtvio/pyTorch/default` - exactly the file and handle `kaggle_reconstruct.ipynb` expects for `KAGGLE_MODEL_HANDLE`.

In [2]:
NC = "Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)"   # Kaggle's exact licence name
APACHE = "Apache 2.0"

# files: {remote name: uploaded name or None to keep it}; omitted = the whole repo.
MODELS = {
    "vggt-1b": dict(repo="facebook/VGGT-1B", files={"model.pt": "vggt1b_model.pt"},
                    handle="vggt1b-rtvio/pyTorch/default", license=NC),
    # No FAIR licence in Kaggle's list: uploaded untagged, with Meta's LICENSE.txt inside the model files.
    "vggt-omega-512": dict(repo="facebook/VGGT-Omega", mirror="wincentIsMe/VGGT-Omega",
                           files={"vggt_omega_1b_512.pt": None, "LICENSE.txt": None},
                           handle="vggt-omega/pyTorch/1b-512", license=None),
    "da3-small": dict(repo="depth-anything/DA3-SMALL", handle="depth-anything-3/pyTorch/small", license=APACHE),
    "da3-base": dict(repo="depth-anything/DA3-BASE", handle="depth-anything-3/pyTorch/base", license=APACHE),
    "da3-large-1.1": dict(repo="depth-anything/DA3-LARGE-1.1", handle="depth-anything-3/pyTorch/large-1-1", license=APACHE),
    "da3-metric-large": dict(repo="depth-anything/DA3METRIC-LARGE", handle="depth-anything-3/pyTorch/metric-large", license=APACHE),
    "mapanything-apache": dict(repo="facebook/map-anything-apache", handle="map-anything/pyTorch/apache", license=APACHE),
}

DATASETS = {
    "mars-lvig-hkairport-gnss03": dict(
        drive_folder="https://drive.google.com/drive/folders/1aG21le4QZl9LhSLE1O0vuVDQ8YgAwdas",
        file="HKairport_GNSS03.bag", size_gb=9.09,
        license="CC BY-NC-SA 4.0 (academic use only) - https://mars.hku.hk/dataset.html"),
}

RUN_MODELS = ["vggt-1b", "da3-small", "da3-base", "da3-large-1.1", "da3-metric-large",
              "mapanything-apache", "vggt-omega-512"]
RUN_DATASETS = ["mars-lvig-hkairport-gnss03"]

# "official" needs HF_TOKEN from an approved account; "mirror" is opt-in (see the note at the top).
VGGT_OMEGA_SOURCE = "official"

## 2. Helpers

In [3]:
import glob, pickletools, shutil, zipfile
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

api = HfApi()


def scratch_root():
    """Largest-free existing scratch dir (not /kaggle/working, whose contents become notebook output)."""
    cands = [p for p in ("/kaggle/temp", "/kaggle/tmp", "/tmp") if os.path.isdir(p)]
    return max(cands, key=lambda p: shutil.disk_usage(p).free)


def ensure_space(nbytes, root):
    free = shutil.disk_usage(root).free
    print("  need %.2f GB, %.2f GB free in %s" % (nbytes / 1e9, free / 1e9, root))
    assert free > nbytes * 1.05 + 1e9, "not enough disk in %s for this item" % root


def pickle_imports(pt_path):
    """(all imports, non-torch imports) a .pt's pickle would perform on load - read with pickletools, never unpickled."""
    with zipfile.ZipFile(pt_path) as z:
        data = z.read(next(n for n in z.namelist() if n.endswith("data.pkl")))
    found, strings = set(), []
    for op, arg, _ in pickletools.genops(data):
        if op.name in ("GLOBAL", "INST"):
            found.add(arg.replace(" ", "."))
        elif op.name == "STACK_GLOBAL" and len(strings) >= 2:
            found.add(strings[-2] + "." + strings[-1])
        if isinstance(arg, str):
            strings.append(arg)
    return sorted(found), sorted(g for g in found if not g.startswith(("torch.", "collections.")))


def fetch_and_upload_model(key):
    m = MODELS[key]
    repo = m["mirror"] if (key == "vggt-omega-512" and VGGT_OMEGA_SOURCE == "mirror") else m["repo"]
    try:
        info = api.model_info(repo, files_metadata=True, token=HF_TOKEN)
    except Exception as e:
        raise RuntimeError("cannot read %s (%s: %s)%s" % (repo, type(e).__name__, e,
            " - request access at https://huggingface.co/%s and add an approved HF_TOKEN secret, "
            "or set VGGT_OMEGA_SOURCE = 'mirror'" % repo if repo == "facebook/VGGT-Omega" else ""))
    sizes = {s.rfilename: s.size or 0 for s in info.siblings}
    wanted = list(m["files"]) if m.get("files") else [f for f in sizes if not f.startswith(".")]
    root = scratch_root()
    ensure_space(sum(sizes.get(f, 0) for f in wanted), root)

    d = os.path.join(root, "upload_" + key)
    os.makedirs(d, exist_ok=True)
    try:
        if m.get("files"):
            for f, rename in m["files"].items():
                p = hf_hub_download(repo, f, local_dir=d, token=HF_TOKEN, revision=info.sha)
                if rename:
                    os.replace(p, os.path.join(d, rename))
        else:
            snapshot_download(repo, local_dir=d, token=HF_TOKEN, revision=info.sha)

        for pt in glob.glob(os.path.join(d, "**", "*.pt"), recursive=True):
            imports, bad = pickle_imports(pt)
            print("  %s pickle imports: %s" % (os.path.basename(pt), ", ".join(imports)))
            assert not bad, "REFUSING to upload %s: its pickle imports %s" % (pt, bad)

        notes = "from https://huggingface.co/%s @ %s" % (repo, info.sha[:12])
        if repo != m["repo"]:
            notes += " (UNOFFICIAL mirror of %s; not byte-identical to it, tensors unverified)" % m["repo"]
        handle = "%s/%s" % (USER, m["handle"])
        kagglehub.model_upload(handle, d, license_name=m["license"], version_notes=notes,
                               ignore_patterns=[".cache/"])
        return handle
    finally:
        shutil.rmtree(d, ignore_errors=True)


def fetch_and_upload_dataset(key):
    import gdown
    ds = DATASETS[key]
    root = scratch_root()
    ensure_space(ds["size_gb"] * 1e9, root)
    # List the folder without downloading it (the whole folder is ~266 GB).
    listing = gdown.download_folder(ds["drive_folder"], skip_download=True, quiet=True) or []
    item = next((f for f in listing if os.path.basename(f.path) == ds["file"]), None)
    assert item is not None, "%s not found in the Drive folder listing" % ds["file"]

    d = os.path.join(root, "upload_" + key)
    os.makedirs(d, exist_ok=True)
    try:
        out = gdown.download(id=item.id, output=os.path.join(d, ds["file"]), quiet=False)
        assert out and os.path.getsize(out) > 0.9 * ds["size_gb"] * 1e9, (
            "Google Drive download failed or was cut short - large shared files hit Drive's download quota; "
            "retry later, or add the file to your own Drive and point gdown at your copy")
        with open(os.path.join(d, "README.md"), "w") as f:
            f.write("%s from the MARS-LVIG dataset (HKU MaRS Lab), %s.\nLicence: %s\n"
                    % (ds["file"], ds["drive_folder"], ds["license"]))
        handle = "%s/%s" % (USER, key)
        kagglehub.dataset_upload(handle, d, version_notes="%s from %s" % (ds["file"], ds["drive_folder"]))
        return handle
    finally:
        shutil.rmtree(d, ignore_errors=True)

## 3. Run

One item at a time; a failure is reported and the rest continue. Hugging Face downloads are fast inside Kaggle; the 9 GB Drive file and the uploads are the slow parts.

In [4]:
results = {}
for key in RUN_MODELS:
    print("\n=== model:", key)
    try:
        results[key] = ("ok", fetch_and_upload_model(key))
    except Exception as e:
        results[key] = ("FAILED", "%s: %s" % (type(e).__name__, e))
        print("  FAILED:", results[key][1])

for key in RUN_DATASETS:
    print("\n=== dataset:", key)
    try:
        results[key] = ("ok", fetch_and_upload_dataset(key))
    except Exception as e:
        results[key] = ("FAILED", "%s: %s" % (type(e).__name__, e))
        print("  FAILED:", results[key][1])

print("\n" + "\n".join("%-28s %-7s %s" % (k, s, v) for k, (s, v) in results.items()))


=== model: vggt-1b
  need 5.03 GB, 1201.67 GB free in /tmp


model.pt: reconstructing file:   0%|          |  0.00B / 5.03GB            

model.pt: downloading bytes:           |  0.00B            

  vggt1b_model.pt pickle imports: collections.OrderedDict, torch.FloatStorage, torch._utils._rebuild_tensor_v2
Uploading Model https://kaggle.com/models/ ggdelhi/vggt1b-rtvio/pyTorch/default ...
Model 'vggt1b-rtvio' does not exist or access is forbidden for user ' ggdelhi'. Creating or handling Model...
  FAILED: BackendError: Permission 'models.create' was denied

=== model: da3-small
  need 0.14 GB, 1201.67 GB free in /tmp


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Uploading Model https://kaggle.com/models/ ggdelhi/depth-anything-3/pyTorch/metric-large ...
Model 'depth-anything-3' does not exist or access is forbidden for user ' ggdelhi'. Creating or handling Model...
  FAILED: BackendError: Permission 'models.create' was denied

=== model: mapanything-apache
  need 4.91 GB, 1201.67 GB free in /tmp


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Uploading Model https://kaggle.com/models/ ggdelhi/map-anything/pyTorch/apache ...
Model 'map-anything' does not exist or access is forbidden for user ' ggdelhi'. Creating or handling Model...
  FAILED: BackendError: Permission 'models.create' was denied

=== model: vggt-omega-512
  need 4.58 GB, 1201.67 GB free in /tmp
  FAILED: GatedRepoError: 401 Client Error. (Request ID: Root=1-6abc1154-285e710c643da4fc53ec40fc;7fb4fb69-5dd4-4585-bf09-55603f3ce062)

Cannot access gated repo for url https://huggingface.co/facebook/VGGT-Omega/resolve/1041e80fc0e911235d3426b0a3d9a81075111a53/vggt_omega_1b_512.pt.
Access to model facebook/VGGT-Omega is restricted. You must have access to it and be authenticated to access it. Please log in.

=== dataset: mars-lvig-hkairport-gnss03
  need 9.09 GB, 1201.67 GB free in /tmp
  FAILED: FileURLRetrievalError: Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you 

## 4. Using the uploads

In any notebook, **Add Input -> Your Work -> Models/Datasets**, or load by handle:

```python
import kagglehub
vggt_dir = kagglehub.model_download("<you>/vggt1b-rtvio/pyTorch/default")   # contains vggt1b_model.pt
bag_dir  = kagglehub.dataset_download("<you>/mars-lvig-hkairport-gnss03")
```

In `kaggle_reconstruct.ipynb`, set `KAGGLE_MODEL_HANDLE = "<you>/vggt1b-rtvio/pyTorch/default"` to skip the Hugging Face download there.

## 5. Optional: verify the VGGT-Omega mirror against the official weights

Needs an `HF_TOKEN` with approved access to `facebook/VGGT-Omega` and ~9.2 GB of scratch disk (both files are memory-mapped, not read into RAM). Compares every tensor; says whether the mirror's weights are identical despite the different file bytes.

In [5]:
COMPARE_MIRROR = False

if COMPARE_MIRROR:
    import torch
    root = scratch_root()
    ensure_space(2 * 4.58e9, root)
    d = os.path.join(root, "compare_vggt_omega")
    try:
        off = hf_hub_download("facebook/VGGT-Omega", "vggt_omega_1b_512.pt", local_dir=os.path.join(d, "official"), token=HF_TOKEN)
        mir = hf_hub_download("wincentIsMe/VGGT-Omega", "vggt_omega_1b_512.pt", local_dir=os.path.join(d, "mirror"))
        for p in (off, mir):
            assert not pickle_imports(p)[1], "unexpected pickle imports in %s" % p
        a = torch.load(off, map_location="cpu", mmap=True, weights_only=True)
        b = torch.load(mir, map_location="cpu", mmap=True, weights_only=True)
        only_a, only_b = sorted(set(a) - set(b)), sorted(set(b) - set(a))
        differ = [k for k in sorted(set(a) & set(b))
                  if a[k].shape != b[k].shape or a[k].dtype != b[k].dtype or not torch.equal(a[k], b[k])]
        print("tensors: official %d, mirror %d" % (len(a), len(b)))
        print("only in official: %s\nonly in mirror: %s\ndiffering tensors: %d %s"
              % (only_a[:10], only_b[:10], len(differ), differ[:10]))
        print("VERDICT:", "weights IDENTICAL" if not (only_a or only_b or differ) else "weights DIFFER - do not use the mirror")
    finally:
        shutil.rmtree(d, ignore_errors=True)